In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("predictive_maintenance.csv")
print(f"Dimensions of dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

Dimensions of dataset: 10000 rows, 10 columns.


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure


In [2]:
# Drop columns that have no mechanical or physical meaning
df_clean = df.drop(columns=['UDI', 'Product ID'])

# Check for missing values
print("Missing values per column:")
print(df_clean.isnull().sum())

# Inspect the 'Failure Type' distribution
print("\nFailure Type Breakdown:")
print(df_clean['Failure Type'].value_counts())

Missing values per column:
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Target                     0
Failure Type               0
dtype: int64

Failure Type Breakdown:
Failure Type
No Failure                  9652
Heat Dissipation Failure     112
Power Failure                 95
Overstrain Failure            78
Tool Wear Failure             45
Random Failures               18
Name: count, dtype: int64


In [3]:
df_clean = df.drop(columns=['UDI', 'Product ID'])

df_clean['Temperature_Difference'] = df_clean['Process temperature [K]'] - df_clean['Air temperature [K]']
df_clean['Power_Index'] = df_clean['Rotational speed [rpm]'] * df_clean['Torque [Nm]']
df_clean['Wear_Efficiency'] = df_clean['Tool wear [min]'] * df_clean['Torque [Nm]']

type_mapping = {'L': 0, 'M': 1, 'H': 2}
df_clean['Type'] = df_clean['Type'].replace(type_mapping)

df_clean.head(10)

C:\Users\chtej\AppData\Local\Temp\ipykernel_19776\2320996593.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean['Type'] = df_clean['Type'].replace(type_mapping)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type,Temperature_Difference,Power_Index,Wear_Efficiency
0,1,298.1,308.6,1551,42.8,0,0,No Failure,10.5,66382.8,0.0
1,0,298.2,308.7,1408,46.3,3,0,No Failure,10.5,65190.4,138.9
2,0,298.1,308.5,1498,49.4,5,0,No Failure,10.4,74001.2,247.0
3,0,298.2,308.6,1433,39.5,7,0,No Failure,10.4,56603.5,276.5
4,0,298.2,308.7,1408,40.0,9,0,No Failure,10.5,56320.0,360.0
5,1,298.1,308.6,1425,41.9,11,0,No Failure,10.5,59707.5,460.9
6,0,298.1,308.6,1558,42.4,14,0,No Failure,10.5,66059.2,593.6
7,0,298.1,308.6,1527,40.2,16,0,No Failure,10.5,61385.4,643.2
8,1,298.3,308.7,1667,28.6,18,0,No Failure,10.4,47676.2,514.8
9,1,298.5,309.0,1741,28.0,21,0,No Failure,10.5,48748.0,588.0


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Separate features (X) and target (y)
# We drop both target columns from X because we don't want the model to 'cheat'
X = df_clean.drop(columns=['Target', 'Failure Type'])

# Encode the text labels in 'Failure Type' to integers (e.g., 'Power Failure' -> 2)
le = LabelEncoder()
y = le.fit_transform(df_clean['Failure Type'])

# Save the class names mapping so we can interpret the results later
class_names = le.classes_
print("Mapped classes:", {i: name for i, name in enumerate(class_names)})

# 2. Split the dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

Mapped classes: {0: 'Heat Dissipation Failure', 1: 'No Failure', 2: 'Overstrain Failure', 3: 'Power Failure', 4: 'Random Failures', 5: 'Tool Wear Failure'}

Training set size: 8000 samples
Testing set size: 2000 samples


In [5]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print("Before SMOTE training distribution:", Counter(y_train))

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Resample only the training data
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("After SMOTE training distribution:", Counter(y_train_balanced))

Before SMOTE training distribution: Counter({np.int64(1): 7722, np.int64(0): 90, np.int64(3): 76, np.int64(2): 62, np.int64(5): 36, np.int64(4): 14})
After SMOTE training distribution: Counter({np.int64(1): 7722, np.int64(5): 7722, np.int64(4): 7722, np.int64(3): 7722, np.int64(0): 7722, np.int64(2): 7722})


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize the model 
# We use the balanced class weights just in case, and set a random state
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train the model using the BALANCED SMOTE data
print("Training the Random Forest model on balanced data...")
rf_model.fit(X_train_balanced, y_train_balanced)

# 3. Make predictions on the real, untouched test data
y_preds = rf_model.predict(X_test)

# 4. Generate the reports using our original text names for clarity!
print("\n------ Classification Report ------")
print(classification_report(y_test, y_preds, target_names=class_names))

print("\n------ Confusion Matrix ------")
print(confusion_matrix(y_test, y_preds))

Training the Random Forest model on balanced data...

------ Classification Report ------
                          precision    recall  f1-score   support

Heat Dissipation Failure       0.96      1.00      0.98        22
              No Failure       0.99      0.99      0.99      1930
      Overstrain Failure       0.94      1.00      0.97        16
           Power Failure       1.00      1.00      1.00        19
         Random Failures       0.00      0.00      0.00         4
       Tool Wear Failure       0.05      0.11      0.07         9

                accuracy                           0.98      2000
               macro avg       0.66      0.68      0.67      2000
            weighted avg       0.99      0.98      0.98      2000


------ Confusion Matrix ------
[[  22    0    0    0    0    0]
 [   1 1904    0    0    7   18]
 [   0    0   16    0    0    0]
 [   0    0    0   19    0    0]
 [   0    4    0    0    0    0]
 [   0    7    1    0    0    1]]


In [7]:
# 1. Filter out 'Random Failures' because they are mathematically unpredictable
df_filtered = df_clean[df_clean['Failure Type'] != 'Random Failures'].copy()

# 2. Re-split features and target
X_filtered = df_filtered.drop(columns=['Target', 'Failure Type'])

# Encode target names cleanly
le_filtered = LabelEncoder()
y_filtered = le_filtered.fit_transform(df_filtered['Failure Type'])
class_names_filtered = le_filtered.classes_

# Train-test split
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

# 3. Re-train Random Forest WITHOUT SMOTE, using internal class balancing instead
rf_balanced = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_balanced.fit(X_train_f, y_train_f)

# 4. Predict and print the real, un-inflated report
y_preds_f = rf_balanced.predict(X_test_f)

print("------ New Cleaned Classification Report ------")
print(classification_report(y_test_f, y_preds_f, target_names=class_names_filtered))

------ New Cleaned Classification Report ------
                          precision    recall  f1-score   support

Heat Dissipation Failure       0.92      1.00      0.96        22
              No Failure       1.00      1.00      1.00      1931
      Overstrain Failure       0.94      1.00      0.97        16
           Power Failure       1.00      1.00      1.00        19
       Tool Wear Failure       0.00      0.00      0.00         9

                accuracy                           0.99      1997
               macro avg       0.77      0.80      0.78      1997
            weighted avg       0.99      0.99      0.99      1997



c:\Users\chtej\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\chtej\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\chtej\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

# 1. Initialize HistGradientBoosting with class balancing enabled
# This model builds sequential trees to specifically target hard-to-classify rows (like Tool Wear)
hgb_model = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)

# 2. Train on our filtered data
print("Training the sequential Gradient Boosting model...")
hgb_model.fit(X_train_f, y_train_f)

# 3. Predict on the test set
y_preds_hgb = hgb_model.predict(X_test_f)

# 4. Print the new report
print("\n------ Gradient Boosting Classification Report ------")
print(classification_report(y_test_f, y_preds_hgb, target_names=class_names_filtered))

Training the sequential Gradient Boosting model...

------ Gradient Boosting Classification Report ------
                          precision    recall  f1-score   support

Heat Dissipation Failure       1.00      1.00      1.00        22
              No Failure       0.99      1.00      1.00      1931
      Overstrain Failure       0.89      1.00      0.94        16
           Power Failure       0.94      0.79      0.86        19
       Tool Wear Failure       0.00      0.00      0.00         9

                accuracy                           0.99      1997
               macro avg       0.76      0.76      0.76      1997
            weighted avg       0.99      0.99      0.99      1997



In [9]:
# 1. Get raw probability predictions from your Random Forest model instead of absolute choices
# This gives us a matrix of shape (samples, number of classes)
y_prob = rf_balanced.predict_proba(X_test_f)

# Find the exact column index for 'Tool Wear Failure'
tool_wear_index = list(class_names_filtered).index('Tool Wear Failure')

# 2. Apply a custom threshold rule:
# If the model thinks there is even a 15% chance of Tool Wear, flag it!
y_custom_preds = []
for prob in y_prob:
    if prob[tool_wear_index] >= 0.15:
        y_custom_preds.append(tool_wear_index)
    else:
        # Otherwise, take the standard highest probability guess
        y_custom_preds.append(np.argmax(prob))

# 3. Print the threshold-adjusted report
print("------ Threshold Adjusted Classification Report ------")
print(classification_report(y_test_f, y_custom_preds, target_names=class_names_filtered))

------ Threshold Adjusted Classification Report ------
                          precision    recall  f1-score   support

Heat Dissipation Failure       0.92      1.00      0.96        22
              No Failure       1.00      1.00      1.00      1931
      Overstrain Failure       0.94      1.00      0.97        16
           Power Failure       1.00      1.00      1.00        19
       Tool Wear Failure       0.12      0.11      0.12         9

                accuracy                           0.99      1997
               macro avg       0.80      0.82      0.81      1997
            weighted avg       0.99      0.99      0.99      1997



In [10]:
# Create a fresh copy to build advanced features
df_advanced = df.drop(columns=['UDI', 'Product ID'])

# 1. Base Physics Features (What we had before)
df_advanced['Temperature_Difference'] = df_advanced['Process temperature [K]'] - df_advanced['Air temperature [K]']
df_advanced['Power_Index'] = df_advanced['Rotational speed [rpm]'] * df_advanced['Torque [Nm]']

# 2. ADVANCED Cumulative Mechanical Stresses
# Friction heat accumulation over time:
df_advanced['Thermal_Strain_Accumulation'] = df_advanced['Tool wear [min]'] * df_advanced['Temperature_Difference']

# Total mechanical work stress over time:
df_advanced['Total_Work_Stress'] = df_advanced['Tool wear [min]'] * df_advanced['Torque [Nm]']

# Safe product quality map
df_advanced['Type'] = df_advanced['Type'].replace({'L': 0, 'M': 1, 'H': 2})

df_advanced[['Tool wear [min]', 'Thermal_Strain_Accumulation', 'Total_Work_Stress']].head()

C:\Users\chtej\AppData\Local\Temp\ipykernel_19776\1016237219.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_advanced['Type'] = df_advanced['Type'].replace({'L': 0, 'M': 1, 'H': 2})


,Tool wear [min],Thermal_Strain_Accumulation,Total_Work_Stress
0,0,0.0,0.0
1,3,31.5,138.9
2,5,52.0,247.0
3,7,72.8,276.5
4,9,94.5,360.0


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Filter out Random Failures
df_adv_filtered = df_advanced[df_advanced['Failure Type'] != 'Random Failures'].copy()

X_adv = df_adv_filtered.drop(columns=['Target', 'Failure Type'])
le_adv = LabelEncoder()
y_adv = le_adv.fit_transform(df_adv_filtered['Failure Type'])
class_names_adv = le_adv.classes_

# Split data
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_adv, y_adv, test_size=0.2, random_state=42, stratify=y_adv
)

# Train the model with heavier weight specifically designed to force focus on tiny classes
rf_advanced = RandomForestClassifier(
    n_estimators=200, 
    max_depth=12,         # Limit depth to prevent it from ignoring rare patterns
    class_weight='balanced_subsample', # Recalculates weights at every tree split
    random_state=42
)
rf_advanced.fit(X_train_a, y_train_a)

# Predict using a slightly optimized threshold for Tool Wear
y_prob_a = rf_advanced.predict_proba(X_test_a)
tool_idx = list(class_names_adv).index('Tool Wear Failure')

y_custom_a = []
for prob in y_prob_a:
    # If the model sees even a 20% signature of cumulative strain, flag it!
    if prob[tool_idx] >= 0.20:
        y_custom_a.append(tool_idx)
    else:
        y_custom_a.append(np.argmax(prob))

print("------ Advanced Cumulative Feature Report ------")
print(classification_report(y_test_a, y_custom_a, target_names=class_names_adv))

------ Advanced Cumulative Feature Report ------
                          precision    recall  f1-score   support

Heat Dissipation Failure       0.91      0.95      0.93        22
              No Failure       1.00      0.95      0.98      1931
      Overstrain Failure       0.94      1.00      0.97        16
           Power Failure       1.00      1.00      1.00        19
       Tool Wear Failure       0.07      0.67      0.12         9

                accuracy                           0.95      1997
               macro avg       0.78      0.92      0.80      1997
            weighted avg       0.99      0.95      0.97      1997



In [12]:
# 1. Install the official IBM Watson Machine Learning SDK
import os
from ibm_watsonx_ai import APIClient
import dotenv

dotenv.load_dotenv()
# 2. Configure your secure credentials
# Replace 'YOUR_IBM_CLOUD_API_KEY' with the key generated in your IBM account under Manage -> IAM -> API Keys
# Replace 'YOUR_WATSON_PROJECT_ID' with the ID found in your Project's 'Manage' tab in Watson Studio
WML_CREDENTIALS = {
    "url": "https://us-south.ml.cloud.ibm.com", # Change to "https://eu-de.ml.cloud.ibm.com" if using Frankfurt
    "apikey": os.getenv("IBM_CLOUD_API_KEY")
}

client = APIClient(WML_CREDENTIALS)

# Set your target workspace to your cloud project
project_id = os.getenv("IBM_PROJECT_ID")
client.set.default_project(project_id)
print("Successfully connected to IBM Watson Machine Learning Space!")

# 3. Define the metadata metadata for your Champion Random Forest Model
model_props = {
    client.repository.ModelMetaNames.NAME: "Predictive_Maintenance_RF_Model",
    client.repository.ModelMetaNames.TYPE: "scikit-learn_1.6", # Matches your scikit-learn environment
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: client.software_specifications.get_id_by_name("runtime-25.1-py3.12")
}

# 4. Save the model into your cloud repository
print("Uploading final optimized model to IBM Cloud...")
model_details = client.repository.store_model(
    model=rf_advanced, # This points directly to your champion linear cumulative model!
    meta_props=model_props,
    training_data=X_train_a,
    training_target=y_train_a
)

print("Model successfully registered! Artifact ID:", client.repository.get_model_id(model_details))

Successfully connected to IBM Watson Machine Learning Space!
Uploading final optimized model to IBM Cloud...
Model successfully registered! Artifact ID: 019e63c7-fb38-749b-aa89-7ed0c62449b6


In [14]:
# Go to the cell right before model training and check this:
print(X_train.columns.tolist())
# This prints the exact data type of every column the model trained on
print(X_train.dtypes)

# This prints a raw sample row exactly how it looks before going to the model
print(X_train.head(1).values.tolist())

['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temperature_Difference', 'Power_Index', 'Wear_Efficiency']
Type                         int64
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Temperature_Difference     float64
Power_Index                float64
Wear_Efficiency            float64
dtype: object
[[1.0, 300.4, 311.8, 1362.0, 47.6, 188.0, 11.400000000000034, 64831.200000000004, 8948.800000000001]]
